In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# from google.colab import files
# uploaded = files.upload("please, upload 'kaggle.json'")

In [3]:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

In [4]:
# # !curl -L \
# #   --limit-rate 5M \ # 메모리 소모를 줄이기 위해 쓰기(?) 제한
# #   --silent --show-error \ # 메모리 소모를 줄이기 위해 진행사항 등 표시 X
# #   -o /content/drive/MyDrive/project_2/datasets/oasis.zip \
# #   https://www.kaggle.com/api/v1/datasets/download/mdfahimbinamin/oasis-1-fastsurfer-quickseg-segmentation-dataset

# # 드라이브 반영까지 오래 걸릴 수 있음
# !curl -L \
#   -o /content/drive/MyDrive/project_2/datasets/oasis.zip \
#   https://www.kaggle.com/api/v1/datasets/download/mdfahimbinamin/oasis-1-fastsurfer-quickseg-segmentation-dataset

In [5]:
!mkdir -p ./oasis1-fastsurfer-quickseg-segmentation-dataset
!unzip /content/drive/MyDrive/project_2/datasets/oasis.zip \
                        -d ./oasis1-fastsurfer-quickseg-segmentation-dataset

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_1/mri/orig/001.mgz  
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_1/mri/orig_nu.mgz  
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_1/scripts/build.log  
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_1/scripts/deep-seg.log  
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_1/stats/aseg+DKT.stats  
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_1/stats/cerebellum.CerebNet.stats  
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_2/mri/aparc.DKTatlas+aseg.deep.mgz  
  inflating: ./oasis1-fastsurfer-quickseg-segmentation-dataset/dataset/CN/OAS1_0386_MR1_2/mri/aseg.auto_noCCseg.mgz  
  inflating: ./oasis1-fastsurfer-quickseg-se

### Setting

In [6]:
!pip -q install nnunetv2 nibabel pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 kB 16.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 5.5 MB/s eta 0:00:00


In [7]:
import os
os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/nnUNet_results"
print(os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"])

/content/nnUNet_raw /content/nnUNet_preprocessed /content/drive/MyDrive/nnUNet_results


### 피질하 중심 재매핑

In [8]:
# SUBCORT_MAP = {
#     # Ventricles / CSF
#     4:  1,   # L Lat Vent
#     43: 2,   # R Lat Vent
#     5:  3,   # L Inf Lat Vent
#     44: 4,   # R Inf Lat Vent
#     14: 5,   # 3rd Vent
#     15: 6,   # 4th Vent
#     24: 7,   # CSF

#     # Subcortical GM
#     10: 8,   # L Thalamus
#     49: 9,   # R Thalamus
#     11: 10,  # L Caudate
#     50: 11,  # R Caudate
#     12: 12,  # L Putamen
#     51: 13,  # R Putamen
#     13: 14,  # L Pallidum
#     52: 15,  # R Pallidum
#     17: 16,  # L Hippocampus
#     53: 17,  # R Hippocampus
#     18: 18,  # L Amygdala
#     54: 19,  # R Amygdala
#     26: 20,  # L Accumbens
#     58: 21,  # R Accumbens

#     # Brainstem
#     16: 22,  # Brain-Stem

#     # Cerebellum (선택이지만 운영에서 도움됨)
#     7:  23,  # L Cerebellum WM
#     46: 24,  # R Cerebellum WM
#     8:  25,  # L Cerebellum Cortex
#     47: 26,  # R Cerebellum Cortex
# }

# LABEL_NAMES = {
#     0:"background",
#     1:"L_LatVent", 2:"R_LatVent", 3:"L_InfLatVent", 4:"R_InfLatVent",
#     5:"V3", 6:"V4", 7:"CSF",
#     8:"L_Thalamus", 9:"R_Thalamus", 10:"L_Caudate", 11:"R_Caudate",
#     12:"L_Putamen", 13:"R_Putamen", 14:"L_Pallidum", 15:"R_Pallidum",
#     16:"L_Hippocampus", 17:"R_Hippocampus", 18:"L_Amygdala", 19:"R_Amygdala",
#     20:"L_Accumbens", 21:"R_Accumbens",
#     22:"BrainStem",
#     23:"L_Cb_WM", 24:"R_Cb_WM", 25:"L_Cb_Ctx", 26:"R_Cb_Ctx"
# }

SUBCORT_MAP = {
    1: 2,    # Left-Cerebral-White-Matter
    2: 4,    # Left-Lateral-Ventricle
    3: 5,    # Left-Inf-Lat-Vent
    4: 7,    # Left-Cerebellum-White-Matter
    5: 8,    # Left-Cerebellum-Cortex
    6: 10,   # Left-Thalamus
    7: 11,   # Left-Caudate
    8: 12,   # Left-Putamen
    9: 13,   # Left-Pallidum
    10: 14,  # 3rd-Ventricle
    11: 15,  # 4th-Ventricle
    12: 16,  # Brain-Stem
    13: 17,  # Left-Hippocampus
    14: 18,  # Left-Amygdala
    15: 24,  # CSF
    16: 26,  # Left-Accumbens-area
    17: 28,  # Left-VentralDC
    18: 31,  # Left-choroid-plexus
    19: 41,  # Right-Cerebral-White-Matter
    20: 43,  # Right-Lateral-Ventricle
    21: 44,  # Right-Inf-Lat-Vent
    22: 46,  # Right-Cerebellum-White-Matter
    23: 47,  # Right-Cerebellum-Cortex
    24: 49,  # Right-Thalamus
    25: 50,  # Right-Caudate
    26: 51,  # Right-Putamen
    27: 52,  # Right-Pallidum
    28: 53,  # Right-Hippocampus
    29: 54,  # Right-Amygdala
}

LABEL_NAMES = {
    0:  "background",

    1:  "L_Cerebral_WM",     # Left-Cerebral-White-Matter
    2:  "L_LatVent",         # Left-Lateral-Ventricle
    3:  "L_InfLatVent",      # Left-Inf-Lat-Vent
    4:  "L_Cb_WM",           # Left-Cerebellum-White-Matter
    5:  "L_Cb_Ctx",          # Left-Cerebellum-Cortex
    6:  "L_Thalamus",        # Left-Thalamus
    7:  "L_Caudate",         # Left-Caudate
    8:  "L_Putamen",         # Left-Putamen
    9:  "L_Pallidum",        # Left-Pallidum
    10: "V3",                # 3rd-Ventricle
    11: "V4",                # 4th-Ventricle
    12: "BrainStem",         # Brain-Stem
    13: "L_Hippocampus",     # Left-Hippocampus
    14: "L_Amygdala",        # Left-Amygdala
    15: "CSF",               # CSF
    16: "L_Accumbens",       # Left-Accumbens-area
    17: "L_VentralDC",       # Left-VentralDC
    18: "L_ChoroidPlexus",   # Left-choroid-plexus

    19: "R_Cerebral_WM",     # Right-Cerebral-White-Matter
    20: "R_LatVent",         # Right-Lateral-Ventricle
    21: "R_InfLatVent",      # Right-Inf-Lat-Vent
    22: "R_Cb_WM",           # Right-Cerebellum-White-Matter
    23: "R_Cb_Ctx",          # Right-Cerebellum-Cortex
    24: "R_Thalamus",        # Right-Thalamus
    25: "R_Caudate",         # Right-Caudate
    26: "R_Putamen",         # Right-Putamen
    27: "R_Pallidum",        # Right-Pallidum
    28: "R_Hippocampus",     # Right-Hippocampus
    29: "R_Amygdala",        # Right-Amygdala
}

In [9]:
import os, re, json
import numpy as np
import nibabel as nib
from tqdm.notebook import tqdm

DATA_ROOT = "/content/oasis1-fastsurfer-quickseg-segmentation-dataset/dataset"
DATASET_ID = 1
DATASET_NAME = "Dataset001_OASIS_Subcort"
OUT_BASE = os.path.join(os.environ["nnUNet_raw"], DATASET_NAME)
imagesTr = os.path.join(OUT_BASE, "imagesTr")
labelsTr = os.path.join(OUT_BASE, "labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

def safe_case_id(path: str) -> str:
    # subject 폴더명을 case id로 쓰기 (문자/숫자/언더바만 남김)
    base = os.path.basename(path.rstrip("/"))
    base = re.sub(r"[^A-Za-z0-9_]+", "_", base)
    return base

def save_nifti_like(src_img: nib.Nifti1Image, data: np.ndarray, out_path: str, dtype=np.float32):
    new_hdr = src_img.header.copy()
    data = data.astype(dtype)
    nib.save(nib.Nifti1Image(data, src_img.affine, new_hdr), out_path)

def remap_labels(label_data: np.ndarray, mapping: dict) -> np.ndarray:
    out = np.zeros(label_data.shape, dtype=np.int16)
    for src, dst in mapping.items():
        out[label_data == int(src)] = int(dst)
    return out

pairs = []
for diag in ["AD", "CN", "MCI"]:
    diag_dir = os.path.join(DATA_ROOT, diag)
    if not os.path.isdir(diag_dir):
        continue
    for subj in os.listdir(diag_dir):
        subj_dir = os.path.join(diag_dir, subj)
        mri_dir = os.path.join(subj_dir, "mri")
        if not os.path.isdir(mri_dir):
            continue

        img_mgz = os.path.join(mri_dir, "orig_nu.mgz")
        lab_mgz = os.path.join(mri_dir, "aseg.auto_noCCseg.mgz")  # 여기로 클래스 줄이기

        if os.path.exists(img_mgz) and os.path.exists(lab_mgz):
            pairs.append((diag, subj_dir, img_mgz, lab_mgz))

print("Found pairs:", len(pairs))
assert len(pairs) > 0, "orig_nu.mgz / aseg.auto_noCCseg.mgz 쌍을 못 찾았어. DATA_ROOT와 구조 확인 필요!"

written = 0
for diag, subj_dir, img_mgz, lab_mgz in tqdm(pairs):
    case = safe_case_id(subj_dir)

    # 1) nnU-Net 규격: .nii.gz
    out_img = os.path.join(imagesTr, f"{case}_0000.nii.gz")
    out_lab = os.path.join(labelsTr, f"{case}.nii.gz")

    if os.path.exists(out_img) and os.path.exists(out_lab):
        continue

    img_nii = nib.load(img_mgz)
    lab_nii = nib.load(lab_mgz)

    img = np.asanyarray(img_nii.dataobj, dtype=np.float32)
    lab = np.asanyarray(lab_nii.dataobj, dtype=np.int32)

    if img.shape != lab.shape:
        print(f"[SKIP] shape mismatch {case}: img{img.shape} vs lab{lab.shape}")
        continue

    # image 저장 (.nii.gz)
    nib.save(
        nib.Nifti1Image(img, img_nii.affine, img_nii.header),
        out_img
    )

    # LUT 재매핑
    max_label = int(lab.max())
    lut = np.zeros(max_label + 1, dtype=np.int16)
    for src, dst in SUBCORT_MAP.items():
        if src <= max_label:
            lut[src] = dst
    lab_remap = lut[lab]

    # label 저장 (.nii.gz)
    nib.save(
        nib.Nifti1Image(lab_remap.astype(np.int16), lab_nii.affine, lab_nii.header),
        out_lab
    )

    # written += 1
    # if written % 50 == 0:
    #     print("written:", written)

print("Done. written:", written)

Found pairs: 1688


  0%|          | 0/1688 [00:00<?, ?it/s]

Done. written: 0


In [10]:
import json, os

dataset_json = {
    "channel_names": {"0": "T1"},
    "labels": {LABEL_NAMES[k]: int(k) for k in sorted(LABEL_NAMES.keys())},
    "numTraining": len([f for f in os.listdir(imagesTr) if f.endswith(".nii.gz")]),
    "file_ending": ".nii.gz"
}

with open(os.path.join(OUT_BASE, "dataset.json"), "w") as f:
    json.dump(dataset_json, f, indent=2)

print("dataset.json saved:", os.path.join(OUT_BASE, "dataset.json"))
print("numTraining:", dataset_json["numTraining"])

dataset.json saved: /content/nnUNet_raw/Dataset001_OASIS_Subcort/dataset.json
numTraining: 1688


In [11]:
import nnunetv2, os, inspect
pkg_dir = os.path.dirname(inspect.getfile(nnunetv2))
pkg_dir

'/usr/local/lib/python3.12/dist-packages/nnunetv2'

In [29]:
import os, textwrap, inspect
import nnunetv2

pkg_dir = os.path.dirname(inspect.getfile(nnunetv2))
trainer_dir = os.path.join(pkg_dir, "training", "nnUNetTrainer")
os.makedirs(trainer_dir, exist_ok=True)

code = r'''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

class nnUNetTrainer_EarlyStop(nnUNetTrainer):
    """
    Patience 기반 early stopping:
    - validation 성능(내부 eval criterion EMA)이 개선되지 않으면 카운트 증가
    - patience 초과 시 학습 종료
    """
    def __init__(self, *args, patience: int = 50, min_delta: float = 1e-6, **kwargs):
        super().__init__(*args, **kwargs)
        self.es_patience = patience
        self.es_min_delta = min_delta
        self._best_ema = None
        self._bad_epochs = 0

    def on_epoch_end(self):
        # 부모 로직(체크포인트 저장/로깅 등)을 그대로 수행
        continue_training = super().on_epoch_end()

        # nnU-Net은 내부적으로 EMA 형태의 평가기준(best_ema 등)을 사용함.
        # v2 구현에 따라 변수명이 조금 다를 수 있어 안전하게 getattr로 접근.
        current_ema = getattr(self, "best_ema", None)
        if current_ema is None:
            # best_ema를 못 찾으면 early stop을 강제하지 않고 계속
            return continue_training

        # best_ema는 "지금까지 best"일 수도 있어서,
        # early stopping 관점에서는 "최근 val 성능 EMA"가 필요할 때가 있는데,
        # 여기선 보수적으로: best_ema가 갱신 안 되면 bad_epochs 증가로 처리.
        if self._best_ema is None:
            self._best_ema = current_ema
            self._bad_epochs = 0
            return continue_training

        # best_ema가 충분히 개선되었는지 체크
        if (current_ema - self._best_ema) > self.es_min_delta:
            self._best_ema = current_ema
            self._bad_epochs = 0
        else:
            self._bad_epochs += 1

        self.print_to_log_file(f"[EarlyStop] bad_epochs={self._bad_epochs}/{self.es_patience}")

        if self._bad_epochs >= self.es_patience:
            self.print_to_log_file("[EarlyStop] Patience exceeded. Stopping training.")
            return False  # training loop stop

        return continue_training
'''
out_path = os.path.join(trainer_dir, "nnUNetTrainer_EarlyStop.py")
with open(out_path, "w") as f:
    f.write(textwrap.dedent(code))

out_path

'/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer_EarlyStop.py'

In [30]:
init_path = os.path.join(trainer_dir, "__init__.py")
if not os.path.exists(init_path):
    with open(init_path, "w") as f:
        f.write("")

with open(init_path, "a") as f:
    f.write("\nfrom .nnUNetTrainer_EarlyStop import nnUNetTrainer_EarlyStop\n")

print("registered:", init_path)

registered: /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/__init__.py


In [32]:
!bash -lc 'for f in /content/nnUNet_raw/Dataset001_OASIS_Subcort/imagesTr/*.nii; do gzip -f "$f"; done'
!bash -lc 'for f in /content/nnUNet_raw/Dataset001_OASIS_Subcort/labelsTr/*.nii; do gzip -f "$f"; done'

In [31]:
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_OASIS_Subcort
Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_plan_and_preprocess", line 8, in <module>
    sys.exit(plan_and_preprocess_entry())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/experiment_planning/plan_and_preprocess_entrypoints.py", line 180, in plan_and_preprocess_entry
    extract_fingerprints(args.d, args.fpe, args.npfp, args.verify_dataset_integrity, args.clean, args.verbose)
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 47, in extract_fingerprints
    extract_fingerprint_dataset(d, fingerprint_extractor_class, num_processes, check_dataset_integrity, clean,
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 30, in extract_fingerprint_dataset
    verify_dataset_integrity(join(nnUNet_raw, dataset_name), num_processes)
  File "/usr/local/li

In [15]:
# 데이터 로드
df = pd.read_csv("/content/oasis1-fastsurfer-quickseg-segmentation-dataset/csv/final_oasis.csv")
df

,IMAGE_ID_x,ID,CDR_x,CDR_y,Unnamed: 4
0,OAS1_0001_MR1_1,OAS1_0001_MR1,0.0,0.0,NaN
1,OAS1_0001_MR1_2,OAS1_0001_MR1,0.0,0.0,NaN
2,OAS1_0001_MR1_3,OAS1_0001_MR1,0.0,0.0,NaN
3,OAS1_0001_MR1_4,OAS1_0001_MR1,0.0,0.0,NaN
4,OAS1_0002_MR1_1,OAS1_0002_MR1,0.0,0.0,NaN
...,...,...,...,...,...
1683,OAS1_0456_MR1_3,OAS1_0456_MR1,0.0,0.0,NaN
1684,OAS1_0456_MR1_4,OAS1_0456_MR1,0.0,0.0,NaN
1685,OAS1_0457_MR1_1,OAS1_0457_MR1,0.0,0.0,NaN
1686,OAS1_0457_MR1_2,OAS1_0457_MR1,0.0,0.0,NaN


In [16]:
df.drop(columns=["CDR_y", "Unnamed: 4"], inplace=True)

In [18]:
df

,IMAGE_ID_x,ID,CDR_x
0,OAS1_0001_MR1_1,OAS1_0001_MR1,0.0
1,OAS1_0001_MR1_2,OAS1_0001_MR1,0.0
2,OAS1_0001_MR1_3,OAS1_0001_MR1,0.0
3,OAS1_0001_MR1_4,OAS1_0001_MR1,0.0
4,OAS1_0002_MR1_1,OAS1_0002_MR1,0.0
...,...,...,...
1683,OAS1_0456_MR1_3,OAS1_0456_MR1,0.0
1684,OAS1_0456_MR1_4,OAS1_0456_MR1,0.0
1685,OAS1_0457_MR1_1,OAS1_0457_MR1,0.0
1686,OAS1_0457_MR1_2,OAS1_0457_MR1,0.0


In [21]:
import re

# MR1_1 로 끝나는 IMAGE_ID_x만 유지
df_mr1_1 = df[
    df['IMAGE_ID_x'].str.match(r'^OAS1_\d+_MR1_1$')
]

df_mr1_1

,IMAGE_ID_x,ID,CDR_x
0,OAS1_0001_MR1_1,OAS1_0001_MR1,0.0
4,OAS1_0002_MR1_1,OAS1_0002_MR1,0.0
8,OAS1_0003_MR1_1,OAS1_0003_MR1,0.5
12,OAS1_0004_MR1_1,OAS1_0004_MR1,0.0
16,OAS1_0005_MR1_1,OAS1_0005_MR1,0.0
...,...,...,...
1672,OAS1_0453_MR1_1,OAS1_0453_MR1,0.5
1675,OAS1_0454_MR1_1,OAS1_0454_MR1,0.5
1678,OAS1_0455_MR1_1,OAS1_0455_MR1,0.0
1681,OAS1_0456_MR1_1,OAS1_0456_MR1,0.0


In [30]:
def mapping(x):
    if x == 0:
        return "CN"
    elif x == 0.5:
        return "MCI"
    elif x == 1.0 or x == 2.0:
        return "AD"
    else:
        np.nan

df_mr1_1.loc[:, "Label"]= df_mr1_1.loc[:, "CDR_x"].apply(mapping)

In [31]:
df_mr1_1["Label"].value_counts()

,count
Label,
CN,316
MCI,70
AD,30


In [33]:
df_mr1_CN = df_mr1_1.loc[df_mr1_1["Label"]=="CN", :]
df_mr1_CN

,IMAGE_ID_x,ID,CDR_x,Label
0,OAS1_0001_MR1_1,OAS1_0001_MR1,0.0,CN
4,OAS1_0002_MR1_1,OAS1_0002_MR1,0.0,CN
12,OAS1_0004_MR1_1,OAS1_0004_MR1,0.0,CN
16,OAS1_0005_MR1_1,OAS1_0005_MR1,0.0,CN
20,OAS1_0006_MR1_1,OAS1_0006_MR1,0.0,CN
...,...,...,...,...
1658,OAS1_0449_MR1_1,OAS1_0449_MR1,0.0,CN
1662,OAS1_0450_MR1_1,OAS1_0450_MR1,0.0,CN
1678,OAS1_0455_MR1_1,OAS1_0455_MR1,0.0,CN
1681,OAS1_0456_MR1_1,OAS1_0456_MR1,0.0,CN


In [37]:
df_mr1_CN_sample100 = df_mr1_CN.sample(n=100, random_state=42).reset_index(drop=True)
df_mr1_CN_sample100

,IMAGE_ID_x,ID,CDR_x,Label
0,OAS1_0246_MR1_1,OAS1_0246_MR1,0.0,CN
1,OAS1_0051_MR1_1,OAS1_0051_MR1,0.0,CN
2,OAS1_0232_MR1_1,OAS1_0232_MR1,0.0,CN
3,OAS1_0110_MR1_1,OAS1_0110_MR1,0.0,CN
4,OAS1_0132_MR1_1,OAS1_0132_MR1,0.0,CN
...,...,...,...,...
95,OAS1_0303_MR1_1,OAS1_0303_MR1,0.0,CN
96,OAS1_0096_MR1_1,OAS1_0096_MR1,0.0,CN
97,OAS1_0180_MR1_1,OAS1_0180_MR1,0.0,CN
98,OAS1_0027_MR1_1,OAS1_0027_MR1,0.0,CN


In [38]:
df_mr1_NCN = df_mr1_1.loc[df_mr1_1["Label"]!="CN", :]
df_mr1_NCN

,IMAGE_ID_x,ID,CDR_x,Label
8,OAS1_0003_MR1_1,OAS1_0003_MR1,0.5,MCI
51,OAS1_0015_MR1_1,OAS1_0015_MR1,0.5,MCI
54,OAS1_0016_MR1_1,OAS1_0016_MR1,0.5,MCI
72,OAS1_0021_MR1_1,OAS1_0021_MR1,0.5,MCI
76,OAS1_0022_MR1_1,OAS1_0022_MR1,0.5,MCI
...,...,...,...,...
1650,OAS1_0447_MR1_1,OAS1_0447_MR1,0.5,MCI
1666,OAS1_0451_MR1_1,OAS1_0451_MR1,0.5,MCI
1669,OAS1_0452_MR1_1,OAS1_0452_MR1,1.0,AD
1672,OAS1_0453_MR1_1,OAS1_0453_MR1,0.5,MCI


In [41]:
df_concat = pd.concat([df_mr1_CN_sample100, df_mr1_NCN], axis=0).reset_index(drop=True)
df_concat

,IMAGE_ID_x,ID,CDR_x,Label
0,OAS1_0246_MR1_1,OAS1_0246_MR1,0.0,CN
1,OAS1_0051_MR1_1,OAS1_0051_MR1,0.0,CN
2,OAS1_0232_MR1_1,OAS1_0232_MR1,0.0,CN
3,OAS1_0110_MR1_1,OAS1_0110_MR1,0.0,CN
4,OAS1_0132_MR1_1,OAS1_0132_MR1,0.0,CN
...,...,...,...,...
195,OAS1_0447_MR1_1,OAS1_0447_MR1,0.5,MCI
196,OAS1_0451_MR1_1,OAS1_0451_MR1,0.5,MCI
197,OAS1_0452_MR1_1,OAS1_0452_MR1,1.0,AD
198,OAS1_0453_MR1_1,OAS1_0453_MR1,0.5,MCI


In [82]:
!rm -rf /content/sampling2/labelsTr

In [86]:
from glob import glob
import shutil
from tqdm.notebook import tqdm
root = "/content/nnUNet_raw/Dataset001_OASIS_Subcort"
dir = ["imagesTr", "labelsTr"]
imagesTr = os.path.join(root, dir[0])
labelsTr = os.path.join(root, dir[1])
IMAGE_ID_x = df_concat.loc[100:, "IMAGE_ID_x"]

for d in dir:
    if d == "imagesTr":
        paths = [os.path.join(root, d, f"{p}_0000.nii.gz") for p in IMAGE_ID_x]
    elif d == "labelsTr":
        paths = [os.path.join(root, d, f"{p}.nii.gz") for p in IMAGE_ID_x]
    for p, f_name in tqdm(zip(paths, IMAGE_ID_x)):
        if not os.path.exists(p):
            print("File Not Exists", p)
        else:
            if d == "imagesTr":
                shutil.copy(p, f"/content/sampling2/{d}/{f_name}_0000.nii.gz")
            elif d == "labelsTr":
                shutil.copy(p, f"/content/sampling2/{d}/{f_name}.nii.gz")


# imagesTr_list = glob(os.path.join(imagesTr, "*.nii.gz"))

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [87]:
import os
os.environ["nnUNet_raw"] = "/content/sampling2"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/nnUNet_results"
print(os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"])

/content/sampling2 /content/nnUNet_preprocessed /content/drive/MyDrive/nnUNet_results


In [88]:
DATASET_ID = 1
CONFIG = "3d_fullres"
FOLD = 0
CHK = "checkpoint_best.pth"

IN_DIR = "/content/sampling2/imagesTr"
OUT_DIR = "/content/infer/preds"

!nnUNetv2_predict \
  -d {DATASET_ID} \
  -i {IN_DIR} \
  -o {OUT_DIR} \
  -c {CONFIG} \
  -f {FOLD} \
  -chk {CHK}


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_predict", line 8, in <module>
    sys.exit(predict_entry_point())
             ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/inference/predict_from_raw_data.py", line 992, in predict_entry_point
    predictor.predict_from_files(args.i, args.o, save_probabilities=args.save_probabilities,
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/inference/predict_from_raw_data.py", line 256, in predict_from_files
    self._manage_input_and_output_lists(list_of_lists_or_source_folder,
  File "/us

In [42]:
df_concat1 = df_concat.iloc[:100, :]
df_concat2 = df_concat.iloc[100:, :]

df_concat1, df_concat2

(         IMAGE_ID_x             ID  CDR_x Label
 0   OAS1_0246_MR1_1  OAS1_0246_MR1    0.0    CN
 1   OAS1_0051_MR1_1  OAS1_0051_MR1    0.0    CN
 2   OAS1_0232_MR1_1  OAS1_0232_MR1    0.0    CN
 3   OAS1_0110_MR1_1  OAS1_0110_MR1    0.0    CN
 4   OAS1_0132_MR1_1  OAS1_0132_MR1    0.0    CN
 ..              ...            ...    ...   ...
 95  OAS1_0303_MR1_1  OAS1_0303_MR1    0.0    CN
 96  OAS1_0096_MR1_1  OAS1_0096_MR1    0.0    CN
 97  OAS1_0180_MR1_1  OAS1_0180_MR1    0.0    CN
 98  OAS1_0027_MR1_1  OAS1_0027_MR1    0.0    CN
 99  OAS1_0259_MR1_1  OAS1_0259_MR1    0.0    CN
 
 [100 rows x 4 columns],
           IMAGE_ID_x             ID  CDR_x Label
 100  OAS1_0003_MR1_1  OAS1_0003_MR1    0.5   MCI
 101  OAS1_0015_MR1_1  OAS1_0015_MR1    0.5   MCI
 102  OAS1_0016_MR1_1  OAS1_0016_MR1    0.5   MCI
 103  OAS1_0021_MR1_1  OAS1_0021_MR1    0.5   MCI
 104  OAS1_0022_MR1_1  OAS1_0022_MR1    0.5   MCI
 ..               ...            ...    ...   ...
 195  OAS1_0447_MR1_1  OAS1_0447_MR

In [43]:
# df_concat1.to_csv("/content/drive/MyDrive/project_2/datasets/concat1.csv", index=False) # 다른 사람이 작업할거임
# df_concat2.to_csv("/content/drive/MyDrive/project_2/datasets/concat2.csv", index=False)

In [70]:
import os
os.environ["nnUNet_raw"] = "/content/sampling2"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/nnUNet_results"
print(os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"])

/content/sampling2 /content/nnUNet_preprocessed /content/drive/MyDrive/nnUNet_results


In [89]:
DATASET_ID=1
CONFIG="3d_fullres"
FOLD=0
CHK="checkpoint_best.pth"   # ← 네가 가진 pt 파일명

IN_DIR="/content/sampling2/imagesTr"
OUT_DIR="/content/infer/preds"

!nnUNetv2_predict -i {IN_DIR} \
                  -o {OUT_DIR} \
                  -d 1 \
                  -c '3d_fullres' \
                  -f 0 \
                  -chk {CHK}


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 100 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 100 cases that I would like to predict

Predicting OAS1_0003_MR1_1:
perform_everything_on_device: True
100% 36/36 [01:23<00:00,  2.31s/it]
sending off prediction to background worker for resampling and export
done with OAS1_0003_MR1_1

Predicting OAS1_0015_MR1_1:
perform_everything_on_device: True
100% 36/36 [01:25<00:00,  2.37s/it]
sending off prediction to background worker for resampling and export
done with OAS1_0015_MR1_1

Predicting OAS1_0016_MR1_1:
perform_ev

In [91]:
len(glob("/content/infer/preds/*.nii.gz"))

100

In [ ]:
import nibabel as nib
import numpy as np

def calculate_brain_ratio(nifti_path, label_dict):
    """
    3D NIfTI 파일을 로드하여 뇌 부위별 상대 용적(Ratio)을 계산하는 함수

    Args:
        nifti_path (str): 추론된 .nii.gz 파일 경로
        label_dict (dict): {라벨번호: 부위명} 매핑 정보

    Returns:
        dict: {부위명: 상대용적비율}
    """
    # 1. NIfTI 이미지 로드
    img = nib.load(nifti_path)
    data = img.get_fdata() # 3D 배열 데이터 (x, y, z)

    # 2. 전체 뇌 용적 계산 (Normalization Factor)
    # 배경(0)을 제외한 모든 복셀의 합 = Total Brain Volume
    total_brain_voxels = np.sum(data > 0)

    if total_brain_voxels == 0:
        return None # 빈 파일 예외 처리

    # 3. 각 부위별 상대 용적 계산 (Ratio)
    results = {'Total_Voxels': total_brain_voxels}

    for label_id, region_name in label_dict.items():
        # 해당 라벨의 복셀 수 계산
        target_voxels = np.sum(data == label_id)

        # 핵심 수식: (타겟 부위 복셀 수) / (전체 뇌 복셀 수)
        ratio = target_voxels / total_brain_voxels
        results[region_name] = ratio

    return results

# ==========================================
# 사용 예시 (모델 고유 라벨 1~29 적용)
# ==========================================
# 주의: 이 nnU-Net 모델은 FreeSurfer 번호가 아닌 1~29번 시퀀스를 사용함
roi_labels = {
    13: "Left-Hippocampus",   # 좌측 해마
    14: "Left-Amygdala",      # 좌측 편도체
    28: "Right-Hippocampus",  # 우측 해마 (수정된 번호)
    29: "Right-Amygdala"      # 우측 편도체 (수정된 번호)
}

# 실행
file_path = "inference_output/OAS1_xxxx.nii.gz"
ratios = calculate_brain_ratio(file_path, roi_labels)

print(f"좌측 해마 비율: {ratios['Left-Hippocampus']:.5f}") # 예: 0.00350
print(f"우측 해마 비율: {ratios['Right-Hippocampus']:.5f}")